[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/03-probability-and-information.ipynb)

# Module 3: Probability and Information

Companion notebook to [Module 3](https://transformers-and-math.drmhse.com/modules/03-probability-and-information) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## Part 1 — Sampling and the normal distribution (section 3.1)

In [ ]:
draws = rng.normal(0, 1, 100_000)
print('within 1 sd:', np.mean(np.abs(draws) < 1).round(3), ' within 2 sd:', np.mean(np.abs(draws) < 2).round(3))

## Part 2 — The rules for expectation and variance (section 3.2)
The variance of a dot product of two random $\mathcal{N}(0,1)$ vectors of length $d$ is $d$. This is the fact behind section 6.3.

In [ ]:
for d in (4, 16, 64, 256):
    q, k = rng.normal(size=(20_000, d)), rng.normal(size=(20_000, d))
    z = (q * k).sum(axis=1)
    print(f'd={d:4}  variance={z.var():8.2f}   variance/d={z.var() / d:.3f}')

# Averages of n samples have standard deviation shrinking like 1/sqrt(n).
for n in (1, 16, 256):
    means = rng.normal(size=(20_000, n)).mean(axis=1)
    print(f'n={n:4}  sd of the average = {means.std():.4f}   1/sqrt(n) = {1 / np.sqrt(n):.4f}')

## Part 3 — Maximum likelihood (section 3.4)
Flip a biased coin, then find the $p$ with the lowest negative log-likelihood by brute force.

In [ ]:
flips = rng.random(50) < 0.65
heads = flips.sum()
ps = np.linspace(0.001, 0.999, 999)
nll = -(heads * np.log(ps) + (len(flips) - heads) * np.log(1 - ps))
print(f'heads {heads}/50 = {heads / 50:.3f};  best p = {ps[nll.argmin()]:.3f}')
plt.plot(ps, nll); plt.xlabel('p'); plt.ylabel('NLL'); plt.show()

## Part 4 — Entropy, cross-entropy, KL, perplexity (section 3.5)

In [ ]:
def H(p): return -np.sum(p * np.log2(p))
def CE(p, q): return -np.sum(p * np.log2(q))

p = np.array([0.5, 0.25, 0.15, 0.10])
for q in (np.full(4, 0.25), np.array([0.4, 0.3, 0.2, 0.1]), p):
    print(f'q={q}:  H(p)={H(p):.3f}  CE={CE(p, q):.3f}  KL={CE(p, q) - H(p):.3f}')
print('perplexity of a uniform guess over 20 tokens:', np.exp(-np.log(1 / 20)))

## Part 5 — Sigmoid and Bradley–Terry (section 3.6)

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
print('P(A preferred), rewards 1.5 vs 0.5:', sigmoid(1.5 - 0.5))
print('add 10 to both rewards:           ', sigmoid(11.5 - 10.5))

# Fit Bradley–Terry ratings from simulated comparisons by gradient ascent.
true_r = np.array([0.0, 1.0, 2.0])
pairs = [(i, j) for i in range(3) for j in range(3) if i != j for _ in range(300)]
wins = [(i, j) if rng.random() < sigmoid(true_r[i] - true_r[j]) else (j, i) for i, j in pairs]
r = np.zeros(3)
for _ in range(500):
    g = np.zeros(3)
    for w, l in wins:
        s = 1 - sigmoid(r[w] - r[l]); g[w] += s; g[l] -= s
    r += 0.01 * g / len(wins) * 3
print('fitted ratings (shifted to start at 0):', (r - r[0]).round(2), ' true:', true_r)

## Your turn
1. What is the entropy, in bits, of $[\tfrac12, \tfrac14, \tfrac14]$? Check with `H`.
2. Show numerically that $\operatorname{Var}(3X) = 9\operatorname{Var}(X)$ for $X \sim \mathcal{N}(0, 1)$.

<details><summary>Answers</summary>

1. 1.5 bits. 2. `(3 * rng.normal(size=100000)).var()` is close to 9.
</details>